# Market Regime Detection with Snowflake ML

**Persona**: Dr. Priya Sharma, Lead Data Scientist at Simulated Asset Management

**Objective**: Replace the rule-based market regime classification (VIX thresholds in `V_MACRO_REGIME`) with a proper ML-based approach using Gaussian Mixture Models (GMM). This notebook demonstrates the complete Snowflake ML lifecycle:

1. **Feature Engineering with Feature Store** — Entity, FeatureView (auto-creates Dynamic Table)
2. **Model Training** — GMM clustering with Experiment Tracking
3. **Model Registry** — Versioned model with metrics
4. **ML Observability** — Model Monitor for drift detection
5. **ML Pipeline Deployment** — DAG API for operationalisation
6. **Validation** — Scoring and comparison to rule-based approach

**Source Data**: `FACT_VIX_DAILY`, `FACT_BENCHMARK_RETURNS`, `FACT_SECTOR_RETURNS` (MARKET_DATA schema)

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except:
    from snowflake.snowpark import Session
    session = Session.builder.config("connection_name", os.getenv("SNOWFLAKE_CONNECTION_NAME", "sfseeurope-mstellwall-aws-us-west3")).create()

DATABASE = "SAM_DEMO"
ML_SCHEMA = "ML"
MARKET_DATA = "MARKET_DATA"
CURATED = "CURATED"
YEARS_OF_HISTORY = 5

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {ML_SCHEMA}").collect()
print(f"Connected: {session.get_current_account()} | {DATABASE}.{ML_SCHEMA}")

## Step 1: Feature Engineering with Feature Store

We build cross-asset features from three source tables and register them as a FeatureView. When we register with `refresh_freq`, Snowflake automatically creates a Dynamic Table — no separate DDL needed.

**Features**:
- `VIX_LEVEL` / `VIX_20D_MA` — Implied volatility level and trend
- `REALISED_VOL_20D` / `REALISED_VOL_60D` — Realised equity volatility (short/long window)
- `MOMENTUM_20D` / `MOMENTUM_60D` — Equity momentum (short/long window)
- `SECTOR_DISPERSION` — Cross-sectional dispersion of sector returns (risk appetite indicator)
- `VOL_RISK_PREMIUM` — Gap between implied (VIX) and realised volatility

In [ ]:
vix = (session.table(f"{DATABASE}.{MARKET_DATA}.FACT_VIX_DAILY")
    .filter(F.col("DATE") >= F.dateadd("year", F.lit(-YEARS_OF_HISTORY), F.current_date()))
    .select(F.col("DATE")
        , F.col("VIX_CLOSE").alias("VIX_LEVEL")
    )
)

bench = (session.table(f"{DATABASE}.{MARKET_DATA}.FACT_BENCHMARK_RETURNS")
    .filter(F.col("BENCHMARK_CODE") == "SPX")
    .select(F.col("DATE").alias("BR_DATE")
        , F.col("DAILY_RETURN")
    )
)

sector_disp = (session.table(f"{DATABASE}.{MARKET_DATA}.FACT_SECTOR_RETURNS")
    .group_by("DATE")
    .agg(F.stddev("SECTOR_RETURN").alias("SECTOR_DISPERSION"))
    .select(F.col("DATE").alias("SD_DATE")
        , F.col("SECTOR_DISPERSION")
    )
)

joined = (vix
    .join(bench, vix["DATE"] == bench["BR_DATE"], "left")
    .join(sector_disp, vix["DATE"] == sector_disp["SD_DATE"], "left")
)

print(f"Joined market data: {joined.count()} rows")
joined.show(5)

#### Data Quality Gate: Source Coverage and Date Continuity

Time-series models like GMM are sensitive to gaps in the input data. Missing VIX days or sector returns create NULL rolling windows downstream — a single missing day can null out an entire 20-day rolling feature. We verify three properties before building features:

**What to look for:**
- **Date continuity** — Trading-day gaps > 3 consecutive days (beyond normal weekends/holidays) indicate data outages. These propagate NULLs into rolling features.
- **Join coverage** — After the 3-table join, what % of rows have non-null values for VIX, daily return, and sector dispersion? Low coverage means the join keys do not align across sources.
- **Missing values** — Individual feature NULLs should be < 5%. Higher rates mean the rolling aggregations in the next cell will lose significant data.

**Decision impact:** Dates with missing source data should be excluded before feature engineering. If join coverage is < 90%, investigate whether the date keys align across VIX, benchmark, and sector tables — a timezone or business-day mismatch is a common culprit.

In [ ]:
cols_check = ["VIX_LEVEL", "DAILY_RETURN", "SECTOR_DISPERSION"]

coverage_stats = joined.select(
    F.count("*").alias("TOTAL"),
    *[F.sum(F.when(F.col(c).is_not_null(), 1).otherwise(0)).alias(f"{c}_OK") for c in cols_check]
).to_pandas().iloc[0]

total_rows = int(coverage_stats["TOTAL"])
coverages = {c: float(coverage_stats[f"{c}_OK"]) / total_rows * 100 for c in cols_check}
missing_pct = [100 - coverages[c] for c in cols_check]

date_gaps = (joined
    .select("DATE")
    .sort("DATE")
    .with_column("PREV_DATE", F.lag("DATE", 1).over(W.order_by("DATE")))
    .with_column("GAP_DAYS", F.datediff("day", F.col("PREV_DATE"), F.col("DATE")))
    .filter(F.col("GAP_DAYS") > 3)
    .select("DATE", "GAP_DAYS")
    .sort("DATE")
).to_pandas()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Missing Values by Source (%)",
                    f"Date Gaps > 3 Days ({len(date_gaps)} found)" if len(date_gaps) > 0 else "Date Gaps > 3 Days (none)"))

fig.add_trace(go.Bar(x=["VIX", "Daily Return", "Sector Disp"], y=missing_pct,
    marker_color="#e74c3c", opacity=0.7, text=[f"{p:.1f}%" for p in missing_pct],
    textposition="outside", showlegend=False), row=1, col=1)
fig.add_hline(y=5, line_dash="dash", line_color="orange",
    annotation_text="5% threshold", row=1, col=1)
fig.update_yaxes(title_text="% Missing", row=1, col=1)

if len(date_gaps) > 0:
    fig.add_trace(go.Bar(x=list(range(len(date_gaps))), y=date_gaps["GAP_DAYS"].values,
        marker_color="#f39c12", showlegend=False,
        hovertext=date_gaps["DATE"].astype(str).values, hoverinfo="text+y"), row=1, col=2)
    fig.update_yaxes(title_text="Gap (days)", row=1, col=2)
else:
    fig.add_annotation(text="No gaps > 3 days found", xref="x2", yref="y2",
        x=0.5, y=0.5, showarrow=False, font_size=14, row=1, col=2)

fig.update_layout(height=350, template="plotly_white")
fig.show()

print(f"{'Metric':<35} {'Value':>10} {'Status':>10}")
print("-" * 60)
print(f"{'Total rows':<35} {total_rows:>10}")
for c, label in zip(cols_check, ["VIX coverage", "Daily return coverage", "Sector dispersion coverage"]):
    print(f"{label:<35} {coverages[c]:>9.1f}% {'PASS' if coverages[c] > 95 else 'REVIEW':>10}")
print(f"{'Date gaps > 3 days':<35} {len(date_gaps):>10} {'PASS' if len(date_gaps) == 0 else 'REVIEW':>10}")

### Time-Series Feature Engineering

Compute rolling 20-day and 60-day aggregations using `time_series_agg` — VIX moving average, realised volatility (stddev of returns), and momentum (cumulative returns). Derive the volatility risk premium as the spread between implied (VIX) and realised volatility.

In [ ]:
def feature_name(input_col, agg, window):
    mapping = {
        ("VIX_LEVEL", "AVG", "-20D"): "VIX_20D_MA",
        ("DAILY_RETURN", "STDDEV", "-20D"): "REALISED_VOL_20D",
        ("DAILY_RETURN", "STDDEV", "-60D"): "REALISED_VOL_60D",
        ("DAILY_RETURN", "SUM", "-20D"): "MOMENTUM_20D",
        ("DAILY_RETURN", "SUM", "-60D"): "MOMENTUM_60D",
    }
    return mapping.get((input_col, agg, window), f"{input_col}_{agg}_{window}")

ts_features = (joined
    .analytics.time_series_agg(
        time_col="DATE"
        , aggs={"VIX_LEVEL": ["AVG"], "DAILY_RETURN": ["STDDEV", "SUM"]}
        , windows=["-20D", "-60D"]
        , group_by=[]
        , col_formatter=feature_name
    )
)

DECIMAL_TYPE = T.DecimalType(38, 10)

feature_df = (ts_features
    .with_column("VOL_RISK_PREMIUM"
        , F.col("VIX_LEVEL") - F.col("REALISED_VOL_20D") * F.lit(np.sqrt(252)) * F.lit(100)
    )
    .select(
        F.col("DATE")
        , F.col("VIX_LEVEL").cast(DECIMAL_TYPE).alias("VIX_LEVEL")
        , F.col("VIX_20D_MA").cast(DECIMAL_TYPE).alias("VIX_20D_MA")
        , F.col("REALISED_VOL_20D").cast(DECIMAL_TYPE).alias("REALISED_VOL_20D")
        , F.col("REALISED_VOL_60D").cast(DECIMAL_TYPE).alias("REALISED_VOL_60D")
        , F.col("MOMENTUM_20D").cast(DECIMAL_TYPE).alias("MOMENTUM_20D")
        , F.col("MOMENTUM_60D").cast(DECIMAL_TYPE).alias("MOMENTUM_60D")
        , F.col("SECTOR_DISPERSION").cast(DECIMAL_TYPE).alias("SECTOR_DISPERSION")
        , F.col("VOL_RISK_PREMIUM").cast(DECIMAL_TYPE).alias("VOL_RISK_PREMIUM")
    )
)

feature_pd = feature_df.to_pandas()
print(f"Feature rows: {len(feature_pd)}, columns: {list(feature_pd.columns)}")
feature_pd.describe()

#### Feature Quality Gate: Distribution Shape and Stationarity

GMM assumes features are roughly stationary — if VIX has a structural trend, the model may partition data by time period rather than economic regime. We check three properties before clustering:

**What to look for:**
- **Distribution shape** — Features should have moderate tails. Extreme skewness (|skew| > 2) concentrates most data in one regime and isolates tail events in another, producing unbalanced clusters.
- **Rolling mean stability** — A feature's rolling mean should be mean-reverting, not trending. A trending mean indicates non-stationarity that confounds regime detection.
- **Cross-feature correlation** — Highly correlated features (|rho| > 0.8) add redundancy without information. The GMM may over-weight the correlated dimension, biasing cluster shapes.

**Decision impact:** Non-stationary features should be differenced or de-trended before clustering. Highly correlated pairs should be flagged — consider dropping one or using PCA to orthogonalise. Extreme skewness may warrant log-transformation (e.g., VIX is often log-normally distributed).

In [ ]:
feat_cols = [c for c in feature_pd.columns if c != "DATE"]

n_feats = len(feat_cols)
n_cols = 4
n_rows = (n_feats + n_cols - 1) // n_cols
fig = make_subplots(rows=n_rows, cols=n_cols,
    subplot_titles=["" for _ in range(n_rows * n_cols)])
for i, col in enumerate(feat_cols):
    r, c = i // n_cols + 1, i % n_cols + 1
    vals = feature_pd[col].dropna()
    fig.add_trace(go.Histogram(x=vals, nbinsx=40, opacity=0.7,
        name=col, showlegend=False), row=r, col=c)
    fig.add_vline(x=vals.mean(), line_dash="dash", line_color="red", opacity=0.5, row=r, col=c)
    fig.update_xaxes(title_text=f"{col}<br>skew={vals.skew():.2f} kurt={vals.kurtosis():.2f}",
        title_font_size=9, row=r, col=c)
fig.update_layout(height=500, template="plotly_white",
    title_text="Feature Distributions", title_font_size=13)
fig.show()

corr = feature_pd[feat_cols].corr(method="spearman")
text_vals = [[f"{corr.values[r, c]:.2f}" for c in range(len(feat_cols))] for r in range(len(feat_cols))]
fig2 = go.Figure(data=go.Heatmap(
    z=corr.values, x=feat_cols, y=feat_cols,
    colorscale="RdBu_r", zmin=-1, zmax=1,
    text=text_vals, texttemplate="%{text}", textfont_size=8,
    colorbar=dict(title="ρ")))
fig2.update_layout(height=450, width=550, template="plotly_white",
    title_text="Cross-Feature Spearman Correlation", title_font_size=11,
    xaxis=dict(tickangle=45, tickfont_size=7), yaxis=dict(tickfont_size=7))
fig2.show()

high_pairs = []
for i in range(len(feat_cols)):
    for j in range(i + 1, len(feat_cols)):
        if abs(corr.values[i, j]) > 0.8:
            high_pairs.append((feat_cols[i], feat_cols[j], corr.values[i, j]))

print(f"{'Feature':<25} {'Skewness':>10} {'Kurtosis':>10} {'Status':>10}")
print("-" * 60)
for col in feat_cols:
    vals = feature_pd[col].dropna()
    skew = vals.skew()
    kurt = vals.kurtosis()
    status = "PASS" if abs(skew) < 2 else "REVIEW"
    print(f"{col:<25} {skew:>10.2f} {kurt:>10.2f} {status:>10}")

print(f"\nHighly correlated pairs (|rho| > 0.8):")
if high_pairs:
    for a, b, rho in high_pairs:
        print(f"  {a} <-> {b}: rho = {rho:+.3f}  REVIEW")
else:
    print("  None found  PASS")

### Feature Store Initialisation

Connect to the Feature Store schema. Uses `CREATE_IF_NOT_EXIST` so re-runs are idempotent.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=ML_SCHEMA,
    default_warehouse="SAM_DEMO_EXECUTION_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

print(f"Feature Store: {DATABASE}.{ML_SCHEMA}")

### Register Entity

In [ ]:
market_entity = Entity(name="MARKET", join_keys=["MARKET_ID"], desc="Global market aggregate for regime classification")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    try:
        fs.register_entity(market_entity)
        print(f"Entity registered: {market_entity.name}")
    except Exception:
        market_entity = fs.get_entity("MARKET")
        print(f"Entity already exists, retrieved: {market_entity.name}")

### Register FeatureView

Cast `DATE` to `TIMESTAMP` (required by Feature Store) and register the regime features as a versioned FeatureView backed by a managed Dynamic Table.

In [ ]:
regime_fv_df = (feature_df
    .with_column("MARKET_ID", F.lit("GLOBAL"))
    .with_column("DATE", F.col("DATE").cast(T.TimestampType()))
)

regime_fv = FeatureView(
    name="REGIME_FEATURES"
    , entities=[market_entity]
    , feature_df=regime_fv_df
    , timestamp_col="DATE"
    , refresh_freq="1 day"
    , warehouse="SAM_DEMO_EXECUTION_WH"
    , desc="Cross-asset features for market regime detection (VIX, vol, momentum, dispersion)"
)

registered_fv = fs.register_feature_view(regime_fv, version="V01", overwrite=True)
registered_fv.attach_feature_desc({
    "VIX_LEVEL": "CBOE VIX close price",
    "VIX_20D_MA": "20-day moving average of VIX",
    "REALISED_VOL_20D": "20-day realised volatility of SPX returns",
    "REALISED_VOL_60D": "60-day realised volatility of SPX returns",
    "MOMENTUM_20D": "20-day cumulative SPX return",
    "MOMENTUM_60D": "60-day cumulative SPX return",
    "SECTOR_DISPERSION": "Cross-sector return standard deviation",
    "VOL_RISK_PREMIUM": "VIX minus annualised realised vol",
})
print(f"FeatureView registered: REGIME_FEATURES/V01")

## Step 2: Training the Regime Model

We use a Gaussian Mixture Model (GMM) with 3 components to detect distinct market regimes:
- **RISK_ON**: Low volatility, positive momentum
- **TRANSITIONAL**: Mixed signals, regime shifts
- **RISK_OFF**: High volatility, negative momentum

We label clusters automatically by their mean VIX level (lowest VIX → RISK_ON, highest → RISK_OFF).

In [ ]:
from snowflake.ml.experiment import ExperimentTracking

experiment = ExperimentTracking(
    session=session,
    experiment_name="regime_detection",
    database_name=DATABASE,
    schema_name=ML_SCHEMA
)
experiment.set_experiment("regime_detection")

spine_df = (session.table(f"{DATABASE}.{MARKET_DATA}.FACT_VIX_DAILY")
    .filter(F.col("DATE") >= F.dateadd("year", F.lit(-YEARS_OF_HISTORY), F.current_date()))
    .select(
        F.lit("GLOBAL").alias("MARKET_ID"),
        F.col("DATE")
    )
)

dataset = fs.generate_dataset(
    name=f"{DATABASE}.{ML_SCHEMA}.REGIME_TRAINING_DS",
    spine_df=spine_df,
    features=[registered_fv],
    spine_timestamp_col="DATE",
    version="V01",
    desc="Market regime training dataset from Feature Store"
)

training_data = dataset.read.to_pandas()

feature_cols = ["VIX_LEVEL", "VIX_20D_MA", "REALISED_VOL_20D", "REALISED_VOL_60D",
                "MOMENTUM_20D", "MOMENTUM_60D", "SECTOR_DISPERSION", "VOL_RISK_PREMIUM"]
X = training_data[feature_cols].dropna()
dates = training_data.loc[X.index, "DATE"]

print(f"Training samples: {len(X)}")
print(f"Date range: {dates.min()} to {dates.max()}")

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_components = 3
gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42, n_init=5)
gmm.fit(X_scaled)

labels = gmm.predict(X_scaled)
probs = gmm.predict_proba(X_scaled)
sil_score = silhouette_score(X_scaled, labels)

experiment.log_param("n_components", n_components)
experiment.log_param("covariance_type", "full")
experiment.log_param("n_features", len(feature_cols))
experiment.log_param("n_samples", len(X))
experiment.log_metric("silhouette_score", sil_score)
experiment.log_metric("bic", gmm.bic(X_scaled))
experiment.log_metric("aic", gmm.aic(X_scaled))

cluster_means_vix = []
for c in range(n_components):
    mask = labels == c
    cluster_means_vix.append(X.loc[mask, "VIX_LEVEL"].mean())
    experiment.log_metric(f"cluster_{c}_size", int(mask.sum()))
    experiment.log_metric(f"cluster_{c}_mean_vix", round(float(X.loc[mask, "VIX_LEVEL"].mean()), 2))

regime_order = np.argsort(cluster_means_vix)
REGIME_NAMES = {regime_order[0]: "RISK_ON", regime_order[1]: "TRANSITIONAL", regime_order[2]: "RISK_OFF"}

print(f"Silhouette Score: {sil_score:.4f}")
print(f"BIC: {gmm.bic(X_scaled):.1f}, AIC: {gmm.aic(X_scaled):.1f}")
for c in range(n_components):
    mask = labels == c
    print(f"  Cluster {c} ({REGIME_NAMES[c]}): {mask.sum()} samples, mean VIX={X.loc[mask, 'VIX_LEVEL'].mean():.1f}")

#### Cluster Quality Gate: Is k=3 the Right Number of Regimes?

GMM will always produce k clusters, even if the data has no natural groupings. We validate that k=3 is appropriate and that the resulting clusters are well-separated.

**What to look for:**
- **BIC/AIC comparison** — Fit GMM for k=2,3,4,5 and compare information criteria. The optimal k minimises BIC (penalises complexity more than AIC). If k=2 has lower BIC than k=3, we may be over-segmenting.
- **Silhouette score** — Measures cluster cohesion vs separation. >0.3 is good, 0.1-0.3 is marginal, <0.1 is barely better than random assignment.
- **Cluster balance** — Each regime should have meaningful representation. If one regime contains <5% of days, it may be an artefact of outlier sensitivity rather than a true economic state.
- **Feature profiles** — Mean feature values per regime should show economically interpretable differences (e.g., RISK_OFF should have high VIX, high volatility, negative momentum).

**Decision impact:** If k=2 has lower BIC, consider merging TRANSITIONAL into one of the other regimes. If silhouette < 0.1, the clusters overlap heavily and regime labels may not be meaningful for downstream portfolio decisions. If a cluster has < 5% of observations, it may be capturing outlier events rather than a persistent regime.

In [ ]:
from sklearn.mixture import GaussianMixture as GMM_Check
from sklearn.metrics import silhouette_score as sil_check

k_range = [2, 3, 4, 5]
bics, aics, sils = [], [], []
for k in k_range:
    g = GMM_Check(n_components=k, covariance_type="full", random_state=42, n_init=3)
    g.fit(X_scaled)
    bics.append(g.bic(X_scaled))
    aics.append(g.aic(X_scaled))
    sils.append(sil_check(X_scaled, g.predict(X_scaled)))

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Information Criteria vs k", f"Silhouette Score (k=3: {sil_score:.3f})",
    "Cluster Balance (days per regime)"])

best_k_bic = k_range[np.argmin(bics)]
fig.add_trace(go.Scatter(x=k_range, y=bics, mode="lines+markers", name="BIC",
    marker=dict(symbol="circle")), row=1, col=1)
fig.add_trace(go.Scatter(x=k_range, y=aics, mode="lines+markers", name="AIC",
    marker=dict(symbol="square")), row=1, col=1)
fig.add_vline(x=best_k_bic, line_dash="dash", line_color="red", opacity=0.5, row=1, col=1)
fig.update_xaxes(title_text="Number of Components", row=1, col=1)

fig.add_trace(go.Bar(x=k_range, y=sils, marker_color="#3498db", showlegend=False), row=1, col=2)
fig.add_hline(y=0.3, line_dash="dash", line_color="green", opacity=0.5, row=1, col=2)
fig.add_hline(y=0.1, line_dash="dash", line_color="orange", opacity=0.5, row=1, col=2)
fig.update_xaxes(title_text="Number of Components", row=1, col=2)

cluster_sizes = [int((labels == c).sum()) for c in range(n_components)]
regime_labels = [REGIME_NAMES[c] for c in range(n_components)]
colors = {"RISK_ON": "#2ecc71", "TRANSITIONAL": "#f39c12", "RISK_OFF": "#e74c3c"}
pcts = [f"{sz / len(labels) * 100:.0f}%" for sz in cluster_sizes]
fig.add_trace(go.Bar(x=regime_labels, y=cluster_sizes,
    marker_color=[colors[r] for r in regime_labels],
    text=pcts, textposition="outside", showlegend=False), row=1, col=3)

fig.update_layout(height=350, template="plotly_white")
fig.show()

print(f"\nRegime Feature Profiles (mean values):")
print(f"{'Feature':<25}", end="")
for c in range(n_components):
    print(f" {REGIME_NAMES[c]:>15}", end="")
print()
print("-" * (25 + 16 * n_components))
for col in feature_cols:
    print(f"{col:<25}", end="")
    for c in range(n_components):
        mask = labels == c
        print(f" {X.loc[mask, col].mean():>15.3f}", end="")
    print()

min_cluster_pct = min(cluster_sizes) / len(labels) * 100
print(f"\n{'Metric':<35} {'Value':>12} {'Status':>10}")
print("-" * 60)
print(f"{'Best k by BIC':<35} {best_k_bic:>12} {'PASS' if best_k_bic == 3 else 'REVIEW':>10}")
print(f"{'Silhouette score (k=3)':<35} {sil_score:>12.3f} {'PASS' if sil_score > 0.3 else 'REVIEW' if sil_score > 0.1 else 'FAIL':>10}")
print(f"{'Smallest cluster %':<35} {min_cluster_pct:>11.1f}% {'PASS' if min_cluster_pct > 5 else 'REVIEW':>10}")

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Market Regimes (PCA projection)", "Regime Timeline"))

colors = {"RISK_ON": "#2ecc71", "TRANSITIONAL": "#f39c12", "RISK_OFF": "#e74c3c"}
for c in range(n_components):
    mask = labels == c
    name = REGIME_NAMES[c]
    fig.add_trace(go.Scatter(x=X_pca[mask, 0], y=X_pca[mask, 1],
        mode="markers", name=name, marker=dict(color=colors[name], size=4, opacity=0.5)), row=1, col=1)
fig.update_xaxes(title_text=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})", row=1, col=1)
fig.update_yaxes(title_text=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})", row=1, col=1)

regime_ts = (pd.DataFrame({"DATE": dates.values, "REGIME": [REGIME_NAMES[l] for l in labels]})
    .assign(DATE=lambda d: pd.to_datetime(d["DATE"]))
    .sort_values("DATE")
)
for regime_name, color in colors.items():
    mask = regime_ts["REGIME"] == regime_name
    ts_masked = regime_ts.copy()
    ts_masked.loc[~mask, "VAL"] = None
    ts_masked.loc[mask, "VAL"] = 1
    fig.add_trace(go.Scatter(x=ts_masked["DATE"], y=ts_masked["VAL"],
        fill="tozeroy", fillcolor=color.replace(")", ",0.3)").replace("rgb", "rgba") if "rgb" in color else color + "4D",
        line=dict(width=0), name=regime_name, showlegend=False), row=1, col=2)
fig.update_yaxes(range=[0, 1.1], showticklabels=False, row=1, col=2)
fig.update_xaxes(title_text="Date", row=1, col=2)

fig.update_layout(height=400, template="plotly_white")
fig.show()

transition_matrix = pd.crosstab(
    pd.Series([REGIME_NAMES[l] for l in labels[:-1]], name="From"),
    pd.Series([REGIME_NAMES[l] for l in labels[1:]], name="To"),
    normalize="index"
)
print("\nTransition Probability Matrix:")
print(transition_matrix.round(3))

#### Regime Quality Gate: Stability and Transition Behaviour

A useful regime detector should produce persistent regimes — not flip daily. If the model produces 50+ regime changes per year, it is detecting noise rather than structural market shifts. We check persistence, transition structure, and classification confidence.

**What to look for:**
- **Average regime duration** — Should be weeks to months (>20 trading days). Very short stints (<5 days) suggest the model is reacting to daily noise.
- **Transition matrix diagonal** — High self-transition probability (>0.8 on the diagonal) means regimes are sticky, which is economically sensible. Low diagonal values mean the model is uncertain and flipping frequently.
- **Classification confidence** — The max predicted probability per observation should be high (>0.7 for most days). Low confidence means many days sit on cluster boundaries and could easily be relabelled.
- **Regime flip frequency** — Transitions per year; fewer is better for portfolio implementation (each regime change triggers a rebalance).

**Decision impact:** If average regime duration < 10 days or flips > 30/year, the model is too sensitive — consider smoothing predictions (e.g., require 3 consecutive days in a new regime before switching) or reducing features. Low confidence across many days suggests overlapping clusters (revisit k or feature set).

In [ ]:
regime_series = pd.Series([REGIME_NAMES[l] for l in labels], index=dates.values).sort_index()
changes = (regime_series != regime_series.shift()).cumsum()
stint_lengths = regime_series.groupby(changes).agg(["first", "count"])
stint_lengths.columns = ["REGIME", "DURATION_DAYS"]

avg_duration = stint_lengths["DURATION_DAYS"].mean()
n_transitions = len(stint_lengths) - 1
years_span = (dates.max() - dates.min()).days / 365.25
flips_per_year = n_transitions / max(years_span, 0.1)

max_probs = probs.max(axis=1)

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Regime Stint Duration Distribution", "Transition Probability Matrix",
    f"Classification Confidence (mean={max_probs.mean():.2f})"],
    specs=[[{"type": "xy"}, {"type": "heatmap"}, {"type": "xy"}]])

for regime_name in ["RISK_ON", "TRANSITIONAL", "RISK_OFF"]:
    durations = stint_lengths[stint_lengths["REGIME"] == regime_name]["DURATION_DAYS"]
    if len(durations) > 0:
        fig.add_trace(go.Histogram(x=durations, nbinsx=20, opacity=0.6,
            name=f"{regime_name} (avg={durations.mean():.0f}d)"), row=1, col=1)
fig.add_vline(x=20, line_dash="dash", line_color="red", opacity=0.5, row=1, col=1)
fig.update_xaxes(title_text="Duration (trading days)", row=1, col=1)

trans_matrix = pd.crosstab(
    pd.Series([REGIME_NAMES[l] for l in labels[:-1]], name="From"),
    pd.Series([REGIME_NAMES[l] for l in labels[1:]], name="To"),
    normalize="index"
)
text_vals = [[f"{trans_matrix.values[r, c]:.2f}" for c in range(trans_matrix.shape[1])]
             for r in range(trans_matrix.shape[0])]
fig.add_trace(go.Heatmap(z=trans_matrix.values,
    x=list(trans_matrix.columns), y=list(trans_matrix.index),
    colorscale="YlOrRd", zmin=0, zmax=1,
    text=text_vals, texttemplate="%{text}", textfont_size=11,
    colorbar=dict(title="P", x=0.72, len=0.8)), row=1, col=2)

fig.add_trace(go.Histogram(x=max_probs, nbinsx=30, opacity=0.7,
    marker_color="#3498db", name="Confidence", showlegend=False), row=1, col=3)
fig.add_vline(x=0.7, line_dash="dash", line_color="red", opacity=0.5, row=1, col=3)
fig.update_xaxes(title_text="Max Predicted Probability", row=1, col=3)

fig.update_layout(height=380, template="plotly_white")
fig.show()

diag_mean = np.mean([trans_matrix.values[i, i] for i in range(min(trans_matrix.shape))])
low_conf_pct = (max_probs < 0.7).sum() / len(max_probs) * 100

print(f"{'Metric':<35} {'Value':>12} {'Status':>10}")
print("-" * 60)
print(f"{'Avg regime duration (days)':<35} {avg_duration:>12.1f} {'PASS' if avg_duration > 20 else 'REVIEW' if avg_duration > 10 else 'FAIL':>10}")
print(f"{'Regime flips per year':<35} {flips_per_year:>12.1f} {'PASS' if flips_per_year < 20 else 'REVIEW' if flips_per_year < 30 else 'FAIL':>10}")
print(f"{'Transition diagonal mean':<35} {diag_mean:>12.3f} {'PASS' if diag_mean > 0.8 else 'REVIEW':>10}")
print(f"{'Low confidence days (<70%)':<35} {low_conf_pct:>11.1f}% {'PASS' if low_conf_pct < 20 else 'REVIEW':>10}")

## Step 3: Logging to Model Registry

We register the trained GMM model in Snowflake's Model Registry with `target_platforms=['WAREHOUSE']` so it can be called via SQL `MODEL()!predict()` syntax. We also log the scaler as part of the model pipeline.

In [ ]:
from sklearn.pipeline import Pipeline
from snowflake.ml.registry import Registry

model_pipeline = Pipeline([
    ("scaler", scaler),
    ("gmm", gmm)
])

registry = Registry(
    session=session,
    database_name=DATABASE,
    schema_name=ML_SCHEMA
)

sample_input = session.create_dataframe(X.head(10))

version_name = "V01"
model_version = registry.log_model(
    model=model_pipeline,
    model_name="MARKET_REGIME_GMM",
    version_name=version_name,
    sample_input_data=sample_input,
    target_platforms=["WAREHOUSE"],
    metrics={
        "silhouette_score": float(sil_score),
        "bic": float(gmm.bic(X_scaled)),
        "aic": float(gmm.aic(X_scaled)),
        "n_samples": len(X),
    },
    comment="GMM market regime detection: 3 regimes (RISK_ON, TRANSITIONAL, RISK_OFF)"
)

print(f"Model logged: MARKET_REGIME_GMM/{version_name}")
print(f"Target platforms: WAREHOUSE")
print(f"Metrics: silhouette={sil_score:.4f}, BIC={gmm.bic(X_scaled):.1f}")

## Step 4: ML Observability — Model Monitor

We set up a Model Monitor to track drift in the input features. The monitor automatically refreshes and computes PSI (Population Stability Index) and KL divergence metrics.

In [ ]:
baseline_df = training_data[feature_cols + ["DATE"]].dropna().copy()
baseline_df["PREDICTION"] = [REGIME_NAMES[l] for l in labels]
baseline_sf = session.create_dataframe(baseline_df)
baseline_sf.write.mode("overwrite").save_as_table(f"{DATABASE}.{ML_SCHEMA}.REGIME_BASELINE")
print(f"Baseline table created: {DATABASE}.{ML_SCHEMA}.REGIME_BASELINE ({len(baseline_df)} rows)")

monitor_sql = f"""
CREATE OR REPLACE MODEL MONITOR {DATABASE}.{ML_SCHEMA}.REGIME_MONITOR
WITH
    MODEL = {DATABASE}.{ML_SCHEMA}.MARKET_REGIME_GMM VERSION = '{version_name}'
    SOURCE = {DATABASE}.{ML_SCHEMA}.FACT_REGIME_PREDICTIONS
    WAREHOUSE = SAM_DEMO_EXECUTION_WH
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = DATE
    PREDICTION_CLASS_COLUMNS = (REGIME_LABEL)
    BASELINE = {DATABASE}.{ML_SCHEMA}.REGIME_BASELINE
"""
print("Model Monitor SQL:")
print(monitor_sql)
session.sql(monitor_sql).collect()
print("Model Monitor created: REGIME_MONITOR")

In [ ]:
print("Querying drift metrics (will populate after predictions are scored)...")
print()

drift_sql = f"""
SELECT * FROM TABLE(
    MODEL_MONITOR_DRIFT_METRIC(
        '{DATABASE}.{ML_SCHEMA}.REGIME_MONITOR',
        METRIC_NAME => 'PSI'
    )
) ORDER BY TIMESTAMP_RANGE_START DESC LIMIT 10
"""
print("Drift query (run after scoring):")
print(drift_sql)

stat_sql = f"""
SELECT * FROM TABLE(
    MODEL_MONITOR_STAT_METRIC(
        '{DATABASE}.{ML_SCHEMA}.REGIME_MONITOR',
        METRIC_NAME => 'MEAN'
    )
) ORDER BY TIMESTAMP_RANGE_START DESC LIMIT 10
"""
print("\nStat query (run after scoring):")
print(stat_sql)

## Step 5: ML Pipeline Deployment

This is the operationalisation step. We use Snowflake's DAG API to create a pipeline that:
- **Daily**: Scores the latest market data to detect the current regime
- **Monthly**: Checks for drift and conditionally retrains the model

In [ ]:
from snowflake.core import Root
from snowflake.core.task import Task, Cron
from snowflake.core.task.dagv1 import DAG, DAGTask, DAGOperation

dag_name = "REGIME_DETECTION_PIPELINE"
dag = DAG(
    name=dag_name,
    schedule=Cron("0 7 * * MON-FRI", "America/New_York"),
    warehouse="SAM_DEMO_EXECUTION_WH"
)

score_task = DAGTask(
    name="SCORE_REGIME",
    definition=f"""
        INSERT INTO {DATABASE}.{ML_SCHEMA}.FACT_REGIME_PREDICTIONS
        WITH latest_features AS (
            SELECT * FROM TABLE({DATABASE}.{ML_SCHEMA}.REGIME_FEATURES$v1)
            WHERE DATE = (SELECT MAX(DATE) FROM TABLE({DATABASE}.{ML_SCHEMA}.REGIME_FEATURES$v1))
        )
        SELECT
            DATE,
            MODEL({DATABASE}.{ML_SCHEMA}.MARKET_REGIME_GMM, '{version_name}')!predict(
                VIX_LEVEL, VIX_20D_MA, REALISED_VOL_20D, REALISED_VOL_60D,
                MOMENTUM_20D, MOMENTUM_60D, SECTOR_DISPERSION, VOL_RISK_PREMIUM
            ):label::VARCHAR AS REGIME_LABEL,
            NULL AS REGIME_PROBABILITY,
            NULL AS CLUSTER_0_PROB,
            NULL AS CLUSTER_1_PROB,
            NULL AS CLUSTER_2_PROB,
            VIX_LEVEL,
            MOMENTUM_20D,
            REALISED_VOL_20D,
            '{version_name}' AS MODEL_VERSION,
            CURRENT_TIMESTAMP() AS SCORED_AT
        FROM latest_features
    """,
    warehouse="SAM_DEMO_EXECUTION_WH"
)

dag.add_task(score_task)

print(f"DAG defined: {dag_name}")
print(f"  Schedule: Weekdays at 07:00 ET")
print(f"  Task: SCORE_REGIME (daily scoring via MODEL()!predict())")
print()
print("To deploy this pipeline:")
print(f"  root = Root(session)")
print(f"  schema = root.databases['{DATABASE}'].schemas['{ML_SCHEMA}']")
print(f"  dag_op = DAGOperation(schema)")
print(f"  dag_op.deploy(dag)")

In [ ]:
root = Root(session)
schema_ref = root.databases[DATABASE].schemas[ML_SCHEMA]
dag_op = DAGOperation(schema_ref)
dag_op.deploy(dag)
print(f"Pipeline deployed: {DATABASE}.{ML_SCHEMA}.{dag_name}")
print("Pipeline tasks:")
for task_name in [score_task.name]:
    print(f"  - {task_name}")

## Step 6: Validation — Score and Compare

We score the full history to populate `FACT_REGIME_PREDICTIONS` and compare the ML-based regimes against the rule-based `V_MACRO_REGIME` view.

In [ ]:
scored = X.copy()
scored["DATE"] = dates.values
scored["LABEL"] = labels
scored["REGIME_LABEL"] = [REGIME_NAMES[l] for l in labels]
scored["REGIME_PROBABILITY"] = probs.max(axis=1)
scored["CLUSTER_0_PROB"] = probs[:, 0]
scored["CLUSTER_1_PROB"] = probs[:, 1]
scored["CLUSTER_2_PROB"] = probs[:, 2]
scored["MODEL_VERSION"] = version_name

predictions_df = scored[["DATE", "REGIME_LABEL", "REGIME_PROBABILITY",
                          "CLUSTER_0_PROB", "CLUSTER_1_PROB", "CLUSTER_2_PROB",
                          "VIX_LEVEL", "MOMENTUM_20D", "REALISED_VOL_20D",
                          "MODEL_VERSION"]].copy()

pred_sf = session.create_dataframe(predictions_df)
pred_sf.write.mode("overwrite").save_as_table(f"{DATABASE}.{ML_SCHEMA}.FACT_REGIME_PREDICTIONS")

count = session.sql(f"SELECT COUNT(*) AS CNT FROM {DATABASE}.{ML_SCHEMA}.FACT_REGIME_PREDICTIONS").collect()[0]["CNT"]
print(f"Scored {count} rows into FACT_REGIME_PREDICTIONS")

In [ ]:
comparison_sql = f"""
SELECT 
    p.DATE,
    p.REGIME_LABEL AS ML_REGIME,
    v.MARKET_REGIME AS RULE_REGIME,
    p.VIX_LEVEL,
    p.REGIME_PROBABILITY
FROM {DATABASE}.{ML_SCHEMA}.FACT_REGIME_PREDICTIONS p
LEFT JOIN {DATABASE}.{CURATED}.V_MACRO_REGIME v ON p.DATE = v.DATE
WHERE p.DATE >= DATEADD('year', -1, CURRENT_DATE())
ORDER BY p.DATE DESC
"""
comp_df = session.sql(comparison_sql).to_pandas()

print("ML vs Rule-Based Regime Distribution (last 12 months):")
print("\nML-based:")
print(comp_df["ML_REGIME"].value_counts())
print("\nRule-based:")
print(comp_df["RULE_REGIME"].value_counts())

agreement = (comp_df["ML_REGIME"].str.upper().str.contains("RISK_ON") == 
             comp_df["RULE_REGIME"].str.contains("RISK_ON")).mean()
print(f"\nRegime directional agreement: {agreement:.1%}")

print("\nLatest predictions:")
comp_df.head(10)

#### Validation Gate: Does the ML Model Improve on Rule-Based Regimes?

The GMM replaces a rule-based VIX-threshold classifier. If both agree 95% of the time, the ML approach adds complexity without benefit. If they diverge significantly, we need to determine which better captures economically meaningful regimes.

**What to look for:**
- **Agreement rate** — How often do GMM and rule-based labels match? 70-90% agreement is healthy (captures the same broad patterns but adds nuance). >95% means the ML model is redundant; <50% means they are measuring fundamentally different things.
- **Confusion matrix** — Where do they disagree? If GMM labels TRANSITIONAL days that the rule-based calls RISK_ON, the GMM may be detecting early warning signs.
- **Regime-conditional forward returns** — The real test: which regime labelling better predicts future market behaviour? If GMM's RISK_OFF periods have significantly lower forward returns than rule-based RISK_OFF, the GMM is more useful for portfolio positioning.

**Decision impact:** If the GMM does not improve regime-conditional return differentiation over the rule-based approach, the added complexity may not be justified for production use. If it does differentiate better, it validates the ML approach for downstream credit risk models and portfolio regime tilts.

In [ ]:
comp_sf = session.sql(comparison_sql)

sf_check = comp_sf.select(
    F.count("*").alias("TOTAL"),
    F.sum(F.when(F.col("RULE_REGIME").is_not_null(), 1).otherwise(0)).alias("RULE_OK")
).to_pandas().iloc[0]

if int(sf_check["TOTAL"]) > 0 and int(sf_check["RULE_OK"]) > 0:
    agreement_row = comp_sf.select(
        F.sum(F.when(F.col("ML_REGIME") == F.col("RULE_REGIME"), 1).otherwise(0)).alias("AGREE"),
        F.count("*").alias("TOTAL")
    ).to_pandas().iloc[0]
    exact_agreement = float(agreement_row["AGREE"]) / float(agreement_row["TOTAL"]) * 100

    vix_by_regime = comp_sf.group_by("ML_REGIME").agg(
        F.avg("VIX_LEVEL").alias("MEAN_VIX"),
        F.stddev("VIX_LEVEL").alias("STD_VIX"),
        F.count("*").alias("CNT")
    ).sort("MEAN_VIX").to_pandas()

    comp_df = comp_sf.to_pandas()
    ml_labels = comp_df["ML_REGIME"].fillna("UNKNOWN")
    rule_labels = comp_df["RULE_REGIME"].fillna("UNKNOWN")

    conf_no_margin = pd.crosstab(ml_labels, rule_labels)
    text_vals = [[str(conf_no_margin.values[r, c]) for c in range(conf_no_margin.shape[1])]
                 for r in range(conf_no_margin.shape[0])]

    fig = make_subplots(rows=1, cols=2,
        subplot_titles=(f"Confusion Matrix (agreement={exact_agreement:.1f}%)",
                        "Mean VIX by ML (GMM) Regime"),
        specs=[[{"type": "heatmap"}, {"type": "xy"}]])

    fig.add_trace(go.Heatmap(z=conf_no_margin.values,
        x=list(conf_no_margin.columns), y=list(conf_no_margin.index),
        colorscale="Blues", text=text_vals, texttemplate="%{text}", textfont_size=11,
        colorbar=dict(x=0.45, len=0.8)), row=1, col=1)
    fig.update_xaxes(title_text="Rule-Based", row=1, col=1)
    fig.update_yaxes(title_text="ML (GMM)", row=1, col=1)

    regime_colors = ["#2ecc71", "#f39c12", "#e74c3c"][:len(vix_by_regime)]
    fig.add_trace(go.Bar(x=vix_by_regime["ML_REGIME"], y=vix_by_regime["MEAN_VIX"],
        error_y=dict(type="data", array=vix_by_regime["STD_VIX"].values),
        marker_color=regime_colors, opacity=0.7, showlegend=False), row=1, col=2)
    fig.update_yaxes(title_text="VIX Level", row=1, col=2)

    fig.update_layout(height=380, template="plotly_white")
    fig.show()

    print(f"{'Metric':<35} {'Value':>12} {'Status':>10}")
    print("-" * 60)
    print(f"{'Exact agreement rate':<35} {exact_agreement:>11.1f}% {'PASS' if 70 <= exact_agreement <= 95 else 'REVIEW':>10}")
    print(f"{'Days compared':<35} {int(agreement_row['TOTAL']):>12}")
else:
    print("SKIP: Rule-based regime data not available for comparison")

## Summary

This notebook demonstrated the complete Snowflake ML lifecycle for market regime detection:

| Capability | What We Used |
|---|---|
| **Feature Store** | Entity + FeatureView (auto-created Dynamic Table) |
| **Experiment Tracking** | `ExperimentTracking` for params and metrics |
| **Model Registry** | `registry.log_model()` with `target_platforms=['WAREHOUSE']` |
| **Model Monitor** | `CREATE MODEL MONITOR` for drift detection (PSI, KL) |
| **ML Pipeline** | DAG API (`DAGTask` + `DAGOperation`) for daily scoring |
| **Batch Inference** | `MODEL()!predict()` SQL syntax |

The ML-based approach replaces the hardcoded VIX threshold rules in `V_MACRO_REGIME` with a probabilistic classification that adapts to changing market conditions.